In [1]:
import h2o
import joblib
import pandas as pd
import numpy as np
import os
import re
import helper
import json

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [2]:
current_dir = os.getcwd()
model_filename = 'model/dl_grid_model_66'
knn_initial_filename = 'D:\LoanLens\model\knn_imputer_model.pkl'
knn_cur_filename = 'D:\LoanLens\model\knn_imputer_model_no_multicol.pkl'
scaler_filename = 'D:\LoanLens\model\scaler_no_multicol.pkl'
#years_in_current_job_filename = 'model/years_in_current_job_mapping.pkl'
purpose_filename = 'model/purpose_mapping.pkl'

model_path = os.path.join(current_dir, model_filename)
knn_initial_path = os.path.join(current_dir, knn_initial_filename)
knn_cur_path = os.path.join(current_dir, knn_cur_filename)
scaler_path = os.path.join(current_dir, scaler_filename)
#years_in_current_job_path = os.path.join(current_dir, years_in_current_job_filename)
purpose_path = os.path.join(current_dir, purpose_filename)

In [3]:
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; Java HotSpot(TM) 64-Bit Server VM (build 24+36-3646, mixed mode, sharing)
  Starting server from D:\user\envs\LoanLens\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\psing\AppData\Local\Temp\tmpr9prb4i_
  JVM stdout: C:\Users\psing\AppData\Local\Temp\tmpr9prb4i_\h2o_psing_started_from_python.out
  JVM stderr: C:\Users\psing\AppData\Local\Temp\tmpr9prb4i_\h2o_psing_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,Asia/Kolkata
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.42.0.2
H2O_cluster_version_age:,"2 years, 3 months and 27 days"
H2O_cluster_name:,H2O_from_python_psing_g5awkz
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,5.965 Gb
H2O_cluster_total_cores:,12
H2O_cluster_allowed_cores:,12
H2O_cluster_status:,"locked, healthy"


In [5]:
model = h2o.load_model(model_path)
knn_initial_model = joblib.load(knn_initial_path)
knn_cur_model = joblib.load(knn_cur_path)
scaler = joblib.load(scaler_path)
#years_in_current_job_mapping = joblib.load(years_in_current_job_path)
purpose_mapping = joblib.load(purpose_path)

In [6]:
test_dict = {'current_loan_amount': 10167,
             'term': 'Short Term',
             'credit_score': 7380.0,
             'years_in_current_job': 3,
             'home_ownership': 'Own Home',
             'annual_income': 42701.0,
             'purpose': 'Debt Consolidation',
             'monthly_debt': 761.51,
             'years_of_credit_history': 25.8,
             'months_since_last_delinquent': 5.5,
             'number_of_open_accounts': 7,
             'number_of_credit_problems': 0,
             'current_credit_balance': 11283,
             'maximum_open_credit': 16954.0,
             'bankruptcies': 0.0,
             'tax_liens': 0.0}

In [7]:
def create_dataframe(data, knn_initial_model, purpose_mapping, knn_cur_model, scaler):
    df = pd.DataFrame([data])

    #clean credit score
    df.loc[df['credit_score'] > 850, 'credit_score'] = df.loc[df['credit_score'] > 850, 'credit_score'] / 10

    #clean home ownership
    df['home_ownership'] = df['home_ownership'].replace('HaveMortgage', 'Home Mortgage')

    #convert string values into lower case and snake case
    df = df.applymap(lambda x: x if not isinstance(x, str) or not helper.has_non_ascii(x) else x.encode('ascii', 'ignore').decode('ascii'))
    categorical_cols = ['term', 'home_ownership', 'purpose']
    for col in categorical_cols:
        df[col] = helper.clean_columns(df[col].tolist())

    #convert term 
    term_dict = {'short_term':0, 'long_term':1}
    df.replace({"term": term_dict}, inplace=True)

    #impute missing values if any
    column_names_to_impute = ['current_loan_amount', 'credit_score', 'years_in_current_job', 'annual_income', 'months_since_last_delinquent', 'maximum_open_credit', 'bankruptcies', 'tax_liens']
    column_with_missing_values = df.columns[df.isnull().any()].tolist()
    imputed = knn_initial_model.transform(df[column_names_to_impute].values)
    data_temp = pd.DataFrame(imputed, columns=column_names_to_impute, index = df.index)
    df[column_with_missing_values] = data_temp[column_with_missing_values]

    #simplify purpose
    df['purpose'] = df['purpose'].map(purpose_mapping)
    df['purpose'].fillna('other', inplace=True)

    #feature engineering
    df['debt_equity_ratio'] = df['monthly_debt'] / df['annual_income']
    df['credit_utilization_ratio'] = df['current_credit_balance'] / df['maximum_open_credit']
    df['is_months_delinquent_missing'] = df['months_since_last_delinquent'].isnull().astype(int)
    df['has_stable_job'] = (df['years_in_current_job'] > 2).astype(int)

    df.drop(['bankruptcies', 'monthly_debt', 'annual_income', 'current_credit_balance', 'years_in_current_job', 'maximum_open_credit', 'months_since_last_delinquent', 'tax_liens'], axis = 1, inplace = True)

    #impute credit_utilization_ratio if needed
    column_with_missing_values = df.columns[df.isnull().any()].tolist()
    if len(column_with_missing_values) > 0:
        column_names_to_impute = ['credit_utilization_ratio']
        df = df.replace([np.inf, -np.inf], np.nan)
        imputed = knn_cur_model.transform(df[column_names_to_impute].values)
        data_temp = pd.DataFrame(imputed, columns=column_names_to_impute, index = df.index)
        df[column_names_to_impute] = data_temp
    else:
        pass

    #one hot encode purpose and home_ownership
    #dummy variable names for purpose and home_ownership in the expected dataframe 
    all_purpose_cols = ['purpose_debt_consolidation', 'purpose_other', 'purpose_personal_loans']
    all_home_cols = ['home_own_home', 'home_rent']

    #one hot encode
    new_dummies_purpose = pd.get_dummies(df['purpose'], prefix='purpose').replace({True: 1, False: 0})
    new_dummies_home = pd.get_dummies(df['home_ownership'], prefix='home').replace({True: 1, False: 0})
    list_dummies_purpose = list(new_dummies_purpose.columns)
    list_dummies_home = list(new_dummies_home.columns)

    #create similar column to expected dataframe
    for col in all_purpose_cols:
        if col not in new_dummies_purpose.columns:
            new_dummies_purpose[col] = 0
    for col in all_home_cols:
        if col not in new_dummies_home.columns:
            new_dummies_home[col] = 0

    #drop first dummies if neccessary
    for col in list_dummies_purpose:
        if col in all_purpose_cols:
            pass
        else:
            new_dummies_purpose.drop(col, axis=1,inplace=True)

    for col in list_dummies_home:
        if col in list(all_home_cols):
            pass
        else:
            new_dummies_home.drop(col, axis=1,inplace=True)

    #change the values into the dummy variables
    df[new_dummies_purpose.columns] = new_dummies_purpose
    df[new_dummies_home.columns] = new_dummies_home

    #drop home_ownership and purpose
    df.drop(["home_ownership", "purpose"], axis=1, inplace = True)

    #standardize numeric values except for binaries and ratios
    cols_to_standardize = ['current_loan_amount', 'credit_score', 'years_of_credit_history', 'number_of_open_accounts', 'number_of_credit_problems']
    data_scaled = scaler.transform(df[cols_to_standardize].values)
    data_temp = pd.DataFrame(data_scaled, columns=cols_to_standardize, index = df.index)
    df[cols_to_standardize] = data_temp

    return df    

In [8]:
df = create_dataframe(test_dict, knn_initial_model, purpose_mapping, knn_cur_model, scaler)

In [9]:
def predict_data(model, hf, threshold_metrics='precision'):
    #predict
    predictions = model.predict(hf)

    #convert to pandas dataframe
    prediction_df = predictions.as_data_frame()

    #applying threshold to the prediction
    loan_given_prob = prediction_df['loan_given'].tolist()[0]
    #loan_refused_prob = prediction_df['loan_refused'].tolist()[0]
    threshold = model.find_threshold_by_max_metric(threshold_metrics)

    loan_prediction = [('loan_given' if loan_given_prob > threshold else 'loan_refused').replace('_', ' ').title()]

    #create prediction dictionary for json
    prediction_dict = {}
    prediction_dict['prediction'] = loan_prediction
    return prediction_dict

In [10]:
hf = h2o.H2OFrame(df)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [11]:
predict_data(model, hf, threshold_metrics='precision')

deeplearning prediction progress: |██████████████████████████████████████████████| (done) 100%


{'prediction': ['Loan Given']}

In [12]:
import requests

In [30]:
url = 'http://localhost:8000/predict'

In [31]:
test_dict = {'current_loan_amount': 10167,
             'term': 'Short Term',
             'credit_score': 7380.0,
             'years_in_current_job': 3,
             'home_ownership': 'Own Home',
             'annual_income': 42701.0,
             'purpose': 'Debt Consolidation',
             'monthly_debt': 761.51,
             'years_of_credit_history': 25.8,
             'months_since_last_delinquent': 5.5,
             'number_of_open_accounts': 7,
             'number_of_credit_problems': 0,
             'current_credit_balance': 11283,
             'maximum_open_credit': 16954.0,
             'bankruptcies': 0.0,
             'tax_liens': 0.0}

test_threshold = {
    'threshold_metrics': 'precision',
}

In [32]:
test_data = {
    'data': test_dict,
    'threshold': test_threshold,
}

In [33]:
response = requests.post(
    url,
    json=test_data,  # Serialize the input_dict as JSON
    headers={"Content-Type": "application/json"}  # Set the appropriate Content-Type header
)

In [34]:
response.json()

{'prediction': ['Loan Given']}

In [48]:
test_dict = {'current_loan_amount': 5167,
             'term': 'Long Term',
             'credit_score': 350.0,
             'years_in_current_job': 10,
             'home_ownership': 'Own Home',
             'annual_income': 127010.0,
             'purpose': 'Debt Consolidation',
             'monthly_debt': 1061.51,
             'years_of_credit_history': 25.8,
             'months_since_last_delinquent': 5.5,
             'number_of_open_accounts': 7,
             'number_of_credit_problems': 0,
             'current_credit_balance': 112833,
             'maximum_open_credit': 16954.0,
             'bankruptcies': 0.0,
             'tax_liens': 0.0}

test_threshold = {
    'threshold_metrics': 'precision',
}

In [49]:
test_data = {
    'data': test_dict,
    'threshold': test_threshold,
}

In [50]:
response = requests.post(
    url,
    json=test_data,  
    headers={"Content-Type": "application/json"}  
)

In [51]:
response.json()

{'prediction': ['Loan Refused']}